# Exploratory Data Analysis

## Project Overview

This notebook explores NovoTech's sales data to identify patterns, trends and business insights through exploratory data analysis and data visualization.

The analysis complements the SQL business report by providing visual evidence and deeper exploration of customer behavior, product performance and sales operations.

---

### Objectives

- Explore sales patterns and customer behavior.
- Identify trends across products and sales channels.
- Visualize key business metrics.
- Support the business insights obtained from the SQL analysis.



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Plot settings
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

In [3]:
# Load datasets

customers = pd.read_csv("../data/processed/customers.csv")
products = pd.read_csv("../data/processed/products.csv")
orders = pd.read_csv("../data/processed/orders.csv")

In [4]:
# Verify datasets have been loaded correctly

print(f"Customers: {customers.shape}")
print(f"Products : {products.shape}")
print(f"Orders   : {orders.shape}")

Customers: (12136, 4)
Products : (10, 3)
Orders   : (12135, 13)


In [6]:
customers.columns

Index(['Customer ID', 'Age', 'Gender', 'Loyalty Member'], dtype='str')

In [7]:
orders.columns

Index(['order_id', 'Customer ID', 'product_id', 'Rating', 'Order Status',
       'Payment Method', 'Total Price', 'Unit Price', 'Quantity',
       'Purchase Date', 'Shipping Type', 'Add-ons Purchased', 'Add-on Total'],
      dtype='str')

## Standardizing Column Names

The original CSV files contain a mixture of naming conventions, including spaces, uppercase letters and partially standardized column names.

To improve code readability and maintain consistency with the MySQL database schema used throughout this project, all column names are standardized using lowercase letters and snake_case.

This preprocessing step simplifies data manipulation and ensures a consistent naming convention across SQL, Python and Power BI.

In [11]:
# Rename columns

customers.columns = [
    "customer_id",
    "age",
    "gender",
    "loyalty_member"
]

products.columns = [
    "product_id",
    "sku",
    "product_type"
]

orders.columns = [
    "order_id",
    "customer_id",
    "product_id",
    "rating",
    "order_status",
    "payment_method",
    "total_price",
    "unit_price",
    "quantity",
    "purchase_date",
    "shipping_type",
    "add_ons_purchased",
    "add_on_total"
]

In [12]:
customers.head()

,customer_id,age,gender,loyalty_member
0,19033,47,Female,No
1,1402,77,Male,No
2,14264,36,Male,No
3,9976,36,Female,No
4,19471,54,Female,No


In [13]:
products.head()

,product_id,sku,product_type
0,1,HDP456,Headphones
1,2,LTP123,Laptop
2,3,SKU1001,Smartphone
3,4,SKU1002,Tablet
4,5,SKU1003,Smartwatch


In [14]:
orders.head()

,order_id,customer_id,product_id,rating,order_status,payment_method,total_price,unit_price,quantity,purchase_date,shipping_type,add_ons_purchased,add_on_total
0,1,19033,10,4,Completed,Bank Transfer,4718.46,786.41,6,2024-09-23,Standard,"Impulse Item, Impulse Item",187.94
1,2,1402,6,2,Cancelled,Paypal,3955.95,791.19,5,2024-09-23,Overnight,"Accessory,Accessory,Extended Warranty",99.39
2,3,14264,8,2,Completed,Credit Card,11396.80,1139.68,10,2024-09-23,Standard,"Impulse Item, Impulse Item",163.65
3,4,9976,4,3,Completed,Paypal,2223.27,247.03,9,2024-09-23,Overnight,Accessory,8.97
4,5,19471,10,3,Cancelled,Credit Card,3145.64,786.41,4,2024-09-23,Same Day,"Extended Warranty, Extended Warranty",145.55


## Dataset Overview

Before performing the exploratory analysis, it is important to understand the structure of each dataset.

This section provides an overview of the available tables, including the number of rows, columns, data types and basic descriptive information.

In [15]:
# Dataset dimensions

print("Customers:", customers.shape)
print("Products :", products.shape)
print("Orders   :", orders.shape)

Customers: (12136, 4)
Products : (10, 3)
Orders   : (12135, 13)


In [16]:
customers.info()
products.info()
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 12136 entries, 0 to 12135
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   customer_id     12136 non-null  int64
 1   age             12136 non-null  int64
 2   gender          12135 non-null  str  
 3   loyalty_member  12136 non-null  str  
dtypes: int64(2), str(2)
memory usage: 379.4 KB
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    10 non-null     int64
 1   sku           10 non-null     str  
 2   product_type  10 non-null     str  
dtypes: int64(1), str(2)
memory usage: 372.0 bytes
<class 'pandas.DataFrame'>
RangeIndex: 12135 entries, 0 to 12134
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           12135 non-null  int64  
 1   cust

### Initial Observations

The datasets were successfully loaded and their structure is consistent with the database design.

Some initial observations include:

- The **customers** table contains one missing value in the `gender` column.
- The **products** table is complete, with no missing values.
- The **orders** table contains missing values in `add_ons_purchased`, which are likely to represent orders without additional products.
- Data types are generally appropriate for the analysis, although the `purchase_date` column will be converted to a datetime format in a later step.

## Data Quality Assessment

Before starting the analysis, it is essential to evaluate the quality of the data.

This section examines missing values, duplicate records and potential data quality issues that could affect the reliability of the analysis.

In [18]:
# Missing values

customers.isnull().sum()
products.isnull().sum()
orders.isnull().sum()

order_id                0
customer_id             0
product_id              0
rating                  0
order_status            0
payment_method          0
total_price             0
unit_price              0
quantity                0
purchase_date           0
shipping_type           0
add_ons_purchased    2939
add_on_total            0
dtype: int64

### Missing Values Assessment

The data quality assessment identified a small number of missing values.

- The **customers** and **products** datasets are almost complete.
- The **orders** dataset contains **2,939 missing values** in the `add_ons_purchased` column.
- No missing values were found in the remaining columns.

Since `add_ons_purchased` describes optional purchases, these missing values are likely to represent orders without additional products. This assumption will be validated during the exploratory analysis.

### Duplicate Records Assessment

Duplicate records can introduce bias into the analysis and lead to incorrect conclusions.

This section verifies whether duplicate rows exist in each dataset before proceeding with the exploratory analysis.

In [19]:
# Check duplicate records

print(f"Customers: {customers.duplicated().sum()}")
print(f"Products : {products.duplicated().sum()}")
print(f"Orders   : {orders.duplicated().sum()}")

Customers: 0
Products : 0
Orders   : 0


### Duplicate Records Assessment

No duplicate records were found in any of the datasets.

This indicates that the data has a good level of integrity and no duplicate rows need to be removed before continuing with the analysis.

## Descriptive Statistics

Descriptive statistics provide an initial understanding of the numerical variables by summarizing their central tendency, dispersion and distribution.

This step helps identify unusual values and better understand the characteristics of the dataset before performing the exploratory analysis.

In [23]:
customers.describe().T

,count,mean,std,min,25%,50%,75%,max
customer_id,12136.0,10444.218194,5619.332689,1000.0,5464.75,9916.5,15416.25,19998.0
age,12136.0,49.123187,18.121530,18.0,33.00,49.0,65.00,80.0


In [24]:
orders.describe().T

,count,mean,std,min,25%,50%,75%,max
order_id,12135.0,6068.000000,3503.217093,1.00,3034.50,6068.00,9101.50,12135.00
customer_id,12135.0,10444.688010,5619.325872,1000.00,5466.00,9917.00,15416.50,19998.00
product_id,12135.0,5.543964,2.865945,1.00,3.00,6.00,8.00,10.00
rating,12135.0,3.103173,1.224066,1.00,2.00,3.00,4.00,5.00
total_price,12135.0,3172.521617,2543.608254,20.75,1083.54,2534.49,4718.46,11396.80
unit_price,12135.0,578.458013,313.199140,20.75,361.18,463.96,791.19,1139.68
quantity,12135.0,5.470375,2.868256,1.00,3.00,5.00,8.00,10.00
add_on_total,12135.0,62.339503,58.018744,0.00,7.95,51.93,94.27,292.77


### Descriptive Statistics

The descriptive statistics provide an initial understanding of the numerical variables in the datasets.

Key observations include:

- Customer ages range from **18 to 80 years**, with an average age of **49.1 years**.
- Order ratings are centered around **3.1 out of 5**, indicating a generally neutral level of customer satisfaction.
- The average order value before add-ons is **€3,172.52**, although order values vary considerably, ranging from **€20.75** to **€11,396.80**.
- Customers purchase an average of **5.47 units** per order.
- Additional purchases generate an average of **€62.34** per order, although some orders include no add-ons while others reach **€292.77**.

Overall, the descriptive statistics suggest a dataset with substantial variability in purchasing behavior, making it suitable for further exploratory analysis.